### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from pathlib import Path


d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
### Read all pdf files in a directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Successfully processed {len(documents)} pages.")

        except Exception as e:
            print(f"Error processing file {pdf_file.name}: {e}")

    print(f"\nFinished processing. Total documents loaded: {len(all_documents)}")            
        
    return all_documents

all_pdf_documents = process_all_pdfs("../data")    

Found 4 PDF files to process.

Processing file: CET6.pdf
Successfully processed 1 pages.

Processing file: 电子信息马俊驰.pdf
Successfully processed 1 pages.

Processing file: 研究生准考证.pdf
Successfully processed 1 pages.

Processing file: 诚信复试承诺书.pdf
Successfully processed 2 pages.

Finished processing. Total documents loaded: 5


In [7]:
all_pdf_documents


[Document(metadata={'producer': 'www.neea.edu.cn', 'creator': 'PyPDF', 'creationdate': '2023-08-28T00:31:13+08:00', 'moddate': '2025-02-26T11:01:51+08:00', 'source': '..\\data\\pdf\\CET6.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'CET6.pdf', 'file_type': 'pdf'}, page_content='说明说 明\n全国大学英语四、六级考试（ CET）是由教育部主办\n的全国统一考试，考试对象为在校大学生。考试内容\n包括听、说、读、写、译等语言技能。\nCET笔试考试时间为每年6 月和12月；CET口试考试\n时间为每年5月和11月。\n考生可登录中国教育考试网（www.neea.edu.cn）查\n询、下载电子成绩报告单或自行办理纸质成绩证明。\n电子成绩报告单和纸质成绩证明与纸质成绩报告单具\n有同等效力。\n1.\n2.\n3.\n大学英语六级口语考试能力描述大学英语六级口语考试能力描述\n优秀\n良好\n合格\n能用英语就一般性话题清晰地阐述自己的\n观点，明确地表达自己的态度；能开展深\n入的讨论，发表具有一定深度的见解。语\n言表达适切，自然流畅。\n能用英语就一般性话题阐述自己的观点，\n表明自己的态度；能开展较深入的讨论。\n语言表达准确连贯。\n能用英语就一般性话题进行交流；能参与\n讨论。语言表达基本准确。\n马俊驰\n13092320020903499X\n河北大学\n电子信息工程学院\n130012242221317\n504 182 207 1152024年12月\n-- --\n--\n242213001003467\nI3HN LEAH GQJX YRL1'),
 Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-03-22T14:27:59+08:00', 'aut

In [ ]:
### Text Splitter get into chunks

def split_documents(documents, chunk_size=200, chunk_overlap=20):
  """Splits a list of documents into smaller chunks."""
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len, # 按字符数计算长度
      separators=["\n\n", "\n", " ", ""],
  )
  split_docs = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

  #exmple of split document
  if split_docs:
    print("\nExample of a split document:")
    print(f"Content: {split_docs[0].page_content[:100]}...")  # Print the first 100 characters
    print(f"Metadata: {split_docs[0].metadata}")

  return split_docs  
  

In [9]:
chunks = split_documents(all_pdf_documents)

chunks

Split 5 documents into 23 chunks.

Example of a split document:
Content: 说明说 明
全国大学英语四、六级考试（ CET）是由教育部主办
的全国统一考试，考试对象为在校大学生。考试内容
包括听、说、读、写、译等语言技能。
CET笔试考试时间为每年6 月和12月；CET口试考...
Metadata: {'producer': 'www.neea.edu.cn', 'creator': 'PyPDF', 'creationdate': '2023-08-28T00:31:13+08:00', 'moddate': '2025-02-26T11:01:51+08:00', 'source': '..\\data\\pdf\\CET6.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'CET6.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.neea.edu.cn', 'creator': 'PyPDF', 'creationdate': '2023-08-28T00:31:13+08:00', 'moddate': '2025-02-26T11:01:51+08:00', 'source': '..\\data\\pdf\\CET6.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'CET6.pdf', 'file_type': 'pdf'}, page_content='说明说 明\n全国大学英语四、六级考试（ CET）是由教育部主办\n的全国统一考试，考试对象为在校大学生。考试内容\n包括听、说、读、写、译等语言技能。\nCET笔试考试时间为每年6 月和12月；CET口试考试\n时间为每年5月和11月。\n考生可登录中国教育考试网（www.neea.edu.cn）查\n询、下载电子成绩报告单或自行办理纸质成绩证明。\n电子成绩报告单和纸质成绩证明与纸质成绩报告单具\n有同等效力。'),
 Document(metadata={'producer': 'www.neea.edu.cn', 'creator': 'PyPDF', 'creationdate': '2023-08-28T00:31:13+08:00', 'moddate': '2025-02-26T11:01:51+08:00', 'source': '..\\data\\pdf\\CET6.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'CET6.pdf', 'file_type': 'pdf'}, page_content='有同等效力。\n1.\n2.\n3.\n大学英语六级口语考试能力描述大学英语六级口语考试能力描述\n优秀\n良好\n合格\n能用英语就一般性话题清晰地阐述自己的\n观点，明确地表达自己的态度；能开展深\n入的讨论，发表具有一定深度的见解。语\n言表达适切，自然流畅。\n能用英语就一般性话题阐述自己的观点，\n表明自己的态度；能开展较深入的讨论。\n语言表达准确

### embedding And vectorStoreDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config  import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

import warnings

# ==================== 解决 Hugging Face 警告（Windows 专用） ====================
# 1. 彻底禁用 symlinks 警告（最有效）
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# 2. 屏蔽 UserWarning，让输出更干净
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

In [ ]:
class EmbeddingManager:
  """Handles embedding generation  using SentenceTransformer."""
  def __init__(self,model_name: str = "shibing624/text2vec-base-chinese"):
    """
    Initializes the embedding manager with a specified model.

    Args:
        model_name (str): The name of the SentenceTransformer model to use for generating embeddings.
    """
    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self):
    """Loads the SentenceTransformer model."""
    try:
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Successfully loaded model: {self.model_name}.Embedding dimension: {self.model.get_embedding_dimension()}")
    except Exception as e:
        print(f"Error loading model {self.model_name}: {e}")  
        raise
    
  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """
    Generates embeddings for a list of texts.

    Args:
        texts (List[str]): A list of strings to generate embeddings for.

    Returns:
        np.ndarray: A numpy array with shape(len(texts), embedding_dimension) 
    """

    if not self.model:
        raise ValueError("Model not loaded. Cannot generate embeddings.")

    print(f"Generating embeddings for {len(texts)} texts...")
    embeddings = self.model.encode(texts, show_progress_bar= True)
    print(f"Generation embeddings completed. Shape: {embeddings.shape}")

    return embeddings
  

### initialize embedding manager 

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: shibing624/text2vec-base-chinese


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3498.27it/s]
BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded model: shibing624/text2vec-base-chinese.Embedding dimension: 768


### VectorStore

In [ ]:
class VectorStore:
  """Manages document embeddings in a chromadb vector store."""

  def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
    """
    Initializes the vector store.

    Args:
        collection_name (str): The name of the ChromaDB collection.
        persist_directory (str): Directory to persist vector store.
    """
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    """Initializes the ChromaDB vector store."""
    try:
      #create persistent ChromaDB client
      os.makedirs(self.persist_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)
      
       # ❗ 先删除旧的 collection
      try:
            self.client.delete_collection(self.collection_name)
            print("Old collection deleted.")
      except Exception:
            pass  # 不存在就忽略

      #create or get collection
      self.collection = self.client.get_or_create_collection(
      name=self.collection_name,
      metadata={"description": "PDF documents embeddings for RAG",
          
                "hnsw:space": "cosine"},
      )

      #check if collection is empty
      print(f"Initialized ChromaDB vector store with collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}") 

    except Exception as e: 
      print(f"Error initializing vector store: {e}")
      raise                                   

  def add_documents(self, documents: List[Any], embeddings: np.ndarray): 
    """
    Adds documents and their embeddings to the vector store.

    Args:
        documents (List[Any]): A list of langchain documents.
        embeddings (np.ndarray): A numpy array of embeddings corresponding to the documents.
    """
    if len(documents) != len(embeddings):
      raise ValueError("The number of documents must match the number of embeddings.")

    print(f"Adding {len(documents)} documents to the vector store...")

    # Prepare data for ChromaDB
    ids = []
    metadatas = []
    documents_text = [] 
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
      # Generate a unique ID for each document
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"            
      ids.append(doc_id)     

      # Prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      # Document content
      documents_text.append(doc.page_content)

      # Embeddings
      embeddings_list.append(embedding.tolist())  

      # Add to collection
    try:
        self.collection.add(
          ids=ids,
          embeddings=embeddings_list,
          metadatas=metadatas,
          documents=documents_text
        )
        print(f"Successfully added {len(documents)} documents to the vector store.")
        print(f"Total documents in collection after addition: {self.collection.count()}")
 
    except Exception as e:
        print(f"Error adding documents to vector store: {e}")
        raise

vector_store = VectorStore()      
vector_store


Old collection deleted.
Initialized ChromaDB vector store with collection: pdf_documents
Existing documents in collection: 0


In [13]:
### Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

### Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### store in vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 23 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.54s/it]

Generation embeddings completed. Shape: (23, 768)
Adding 23 documents to the vector store...
Successfully added 23 documents to the vector store.
Total documents in collection after addition: 23


### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
  """Handles query-based retrieval from a vector store."""

  def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
    """
    Initializes the RAG retriever.

    Args:
        vectorstore (VectorStore): The vector store containing document embeddings.
        embedding_manager (EmbeddingManager): Manager for generating query embeddings.
    """
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    """
        Retrieves relevant documents 

        Args:
            query (str): The query to retrieve documents for.
            top_k (int): The number of top results to retrieve.
            score_threshold (float): Minimum similarity score threshold.

        Returns:
            List of dictionaries containing retrieved documents and their metadata.
    """
    print(f"Retrieving documents for query: {query}")
    print(f"Top K: {top_k}, score threshold: {score_threshold}")

    # Generate query embedding
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    # Search in vector store
    try:
        results = self.vector_store.collection.query(
          query_embeddings=[query_embedding.tolist()],
          n_results=top_k,
          )
        
        # Process results 
        retrieve_docs = []
        if results['documents'] and results['documents'][0]:
           documents = results['documents'][0]
           metadatas = results['metadatas'][0]
           distances = results['distances'][0]
           ids = results['ids'][0]

           for i , (doc_id,document, metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):
              # Convert distance to similarity score 
              similarity_score = 1 - distance
              if similarity_score >= score_threshold:
                 retrieve_docs.append({
                    'id': doc_id,
                    'content': document,
                    'metadata': metadata,
                    'similarity_score': similarity_score,
                    'distance': distance,
                    'rank': i + 1, 
                    })
                 
           print(f"distances: {distance},Retrieved {len(retrieve_docs)} documents")   
        else:
           print(f"No documents found")

        return retrieve_docs

    except Exception as e:
       print(f"Error during retrieval: {e}")
       return []   

rag_retriever = RAGRetriever(vector_store,embedding_manager)

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("cet")

Retrieving documents for query: cet
Top K: 3, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.25it/s]

Generation embeddings completed. Shape: (1, 768)
distances: 0.5772413015365601,Retrieved 3 documents


[{'id': 'doc_c20f476b_0',
  'content': '说明说 明\n全国大学英语四、六级考试（ CET）是由教育部主办\n的全国统一考试，考试对象为在校大学生。考试内容\n包括听、说、读、写、译等语言技能。\nCET笔试考试时间为每年6 月和12月；CET口试考试\n时间为每年5月和11月。\n考生可登录中国教育考试网（www.neea.edu.cn）查\n询、下载电子成绩报告单或自行办理纸质成绩证明。\n电子成绩报告单和纸质成绩证明与纸质成绩报告单具\n有同等效力。',
  'metadata': {'producer': 'www.neea.edu.cn',
   'page_label': '1',
   'source': '..\\data\\pdf\\CET6.pdf',
   'moddate': '2025-02-26T11:01:51+08:00',
   'content_length': 200,
   'file_type': 'pdf',
   'creationdate': '2023-08-28T00:31:13+08:00',
   'page': 0,
   'creator': 'PyPDF',
   'total_pages': 1,
   'doc_index': 0,
   'source_file': 'CET6.pdf'},
  'similarity_score': 0.5220701694488525,
  'distance': 0.47792983055114746,
  'rank': 1},
 {'id': 'doc_9d47ee92_5',
  'content': '平均绩点：3.14/5 \n \nJAVA 基础与 Android 开发：84 \n \n积极参加校内活动如辩论赛，志愿活动等。 两次获得河北大学学业奖学金。 \n \n中文成绩单.pdf \n \n技能证书 \n \n \n英语四级 446 分，英语六级 504 分，良好的听说读写能力。 \n \n自我评价 \n \n \n为人沉稳，心态好，能和身边的人很好相处。学习勤奋，做事认真负责。',
  'metadata': {'file_type': 'pdf',
   'source_file': '电子信息马俊驰.

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Deepseek LLM

from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv

load_dotenv(override=True)

### Initialize Deepseek LLM
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
llm = ChatDeepSeek(
  api_key=deepseek_api_key, model="deepseek-chat", 
  temperature=0.1,
  max_tokens=1000
  )

## 2. Simple RAG function: retrieve context and generate response
def rag_simple(query, retriever, llm, top_k=3,):
    ## retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant documents found."

    ## generate response
    prompt = """Use the following context to answer the question concisely.
         Context:
         {context}   

         question: {query}

         Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content


In [18]:
answer = rag_simple("复试不能做什么", rag_retriever, llm)
print(answer)

Retrieving documents for query: 复试不能做什么
Top K: 3, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.26it/s]

Generation embeddings completed. Shape: (1, 768)
distances: 0.5131900310516357,Retrieved 3 documents


根据上下文，复试中不能做以下事情：

1. 由他人协助或传递答案。
2. 拍照、录影录像、截屏。
3. 以任何形式发布复试过程或试题等保密资料。
4. 使用任何设备作弊。
5. 向他人传播考试内容和安排。


#### Enhanced RAG Pipeline Features

In [19]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
  """
  RAG pipeline with extra features:
  - Returns answer, sources, confidence score, and optional full context 
  """
  results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

  if not results:
    return {'answer': 'No relevant documents found.', 'sources': [], 'confidence_score': 0.0, 'context': ''}

  # Prepare context and sources
  context = "\n\n".join([doc['content'] for doc in results])
  sources = [{
     'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
     'page': doc['metadata'].get('page', 'Unknown'),
     'score': doc['similarity_score'],
     'preview': doc['content'][:100]+'...'
    } for doc in results
    ]
  confidence = max([doc['similarity_score'] for doc in results])

  prompt = """Use the following context to answer the question concisely.
    Context:
    {context}

    question: {query}

    Answer:"""

  response = llm.invoke(prompt.format(context=context, query=query))

  output = {
    'answer': response.content,
    'sources': sources,
    'confidence_score': confidence,
  }

  if return_context:
    output['context'] = context

  return output

# use
result = rag_advanced("cet考试要求", rag_retriever, llm, top_k=3, min_score= 0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])

    

Retrieving documents for query: cet考试要求
Top K: 3, score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.46it/s]

Generation embeddings completed. Shape: (1, 768)
distances: 0.5858443379402161,Retrieved 3 documents


Answer: CET考试要求包括：笔试每年6月和12月举行，口试每年5月和11月举行。成绩可通过中国教育考试网查询或下载电子成绩单，电子与纸质成绩单具有同等效力。口试能力分为优秀、良好、合格等级，对应不同的英语表达与讨论能力。考生需按规定准备考试，避免影响正常参加。
Sources: [{'source': 'CET6.pdf', 'page': 0, 'score': 0.6962330341339111, 'preview': '说明说 明\n全国大学英语四、六级考试（ CET）是由教育部主办\n的全国统一考试，考试对象为在校大学生。考试内容\n包括听、说、读、写、译等语言技能。\nCET笔试考试时间为每年6 月和12月；CET口试考...'}, {'source': 'CET6.pdf', 'page': 0, 'score': 0.4229527711868286, 'preview': '有同等效力。\n1.\n2.\n3.\n大学英语六级口语考试能力描述大学英语六级口语考试能力描述\n优秀\n良好\n合格\n能用英语就一般性话题清晰地阐述自己的\n观点，明确地表达自己的态度；能开展深\n入的讨论，发表具...'}, {'source': '研究生准考证.pdf', 'page': 0, 'score': 0.41415566205978394, 'preview': '等公告，按要求做好各项准备。否则，可能影响你正常考试。\n3.第四单元特殊科目(12月22日下午)的考试，考生可登陆保定市教育考试院\n官网(www.bdksy.cn)查询。\n招生单位说明 无\n考  生 ...'}]


In [ ]:
# Advanced RAG Pipeline: streaming, History, Summarization
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
  def __init__(self, retriever, llm):
    self.retriever = retriever
    self.llm = llm
    self.history = []

  def query(self, question: str, top_k: int = 3, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
    # Retrieve relevant documents
    results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

    if not results:
      answer = "No relevant documents found."
      sources = []
      context = ""
    else:
      context = "\n\n".join([doc['content'] for doc in results])
      sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page': doc['metadata'].get('page', 'Unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:100]+'...'
      }for doc in results]
      # Streaming answer simulation
      prompt = """Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"""
      if stream:
        for i in range(0, len(prompt),50):
          print(prompt[i:i+50], end = '', flush=True)
          time.sleep(0.05)
        print()
      response = self.llm.invoke(prompt.format(context=context, question=question))
      answer = response.content

    # Add citation to answer
    citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
    answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

    # Optionally summarize answer:
    summary = None
    if summarize and answer:
      summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
      summary_resp = self.llm.invoke([summary_prompt])
      summary = summary_resp.content  

    # Store query history
    self.history.append({
        'question': question,
        'answer': answer,
        'sources': sources,
        'summary': summary,
        'history': self.history
    }) 

    return {
         'question': question,
         'answer': answer_with_citations,
         'sources': sources,
         'summary': summary,
         'history': self.history,
        
    }     
          
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("马俊驰", top_k = 3, min_score = 0.1, summarize = True, stream=True)
print("\nFinal Answer:", result['answer'])
print("History:", result['history'][-1])
print("Summary:", result['summary'])

Retrieving documents for query: 马俊驰
Top K: 3, score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.14it/s]

Generation embeddings completed. Shape: (1, 768)
distances: 0.5504425764083862,Retrieved 3 documents
Use the following context to answer the question concisely.
Context:
{context}

Question: {question}
Answer:



Final Answer: 马俊驰，河北大学电子信息工程学院学生，平均绩点3.14/5，报考电子信息专业。英语六级504分，具备良好的听说读写能力。曾两次获得学业奖学金，积极参加校内活动。自我评价为人沉稳、学习勤奋、认真负责。

Citations:
[1] CET6.pdf (page 0)
[2] 电子信息马俊驰.pdf (page 0)
[3] 电子信息马俊驰.pdf (page 0)
History: {'question': '马俊驰', 'answer': '马俊驰，河北大学电子信息工程学院学生，平均绩点3.14/5，报考电子信息专业。英语六级504分，具备良好的听说读写能力。曾两次获得学业奖学金，积极参加校内活动。自我评价为人沉稳、学习勤奋、认真负责。', 'sources': [{'source': 'CET6.pdf', 'page': 0, 'score': 0.6042359471321106, 'preview': '能用英语就一般性话题进行交流；能参与\n讨论。语言表达基本准确。\n马俊驰\n13092320020903499X\n河北大学\n电子信息工程学院\n130012242221317\n504 182 207 115...'}, {'source': '电子信息马俊驰.pdf', 'page': 0, 'score': 0.5843520164489746, 'preview': '排名： 21 \n自动控制原理：81 C 程序设计：93 Python 语言程序设计：98 \n姓 名 马俊驰 \n \n民 族 汉 \n \n电话/微信 15612730815 \n \n邮 箱 351008448...'}, {'source': '电子信息马俊驰.pdf', 'page': 0, 'score': 0.44955742359161377, 'preview': '平均绩点：3.14/5 \n \nJAVA 基础与 Android 开发：84 \n \n积极参加校内活动如辩论赛，志愿活动等。 两次获得河北大学学业奖学金。 \n \n中文成绩单.pdf \n \n技能证书 \n \n...'}], 'summary': '马俊驰是河北大学电子信息工程学院学生，报考电子信息专业，平均绩点3.14/5且英语六级504分。他曾两次获得学业